In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset
import numpy as np
from tqdm import tqdm
import copy
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons, make_blobs
import math
import seaborn as sns
from sklearn.datasets import (
    make_blobs,
    make_moons,
    make_swiss_roll,
    make_circles,
    make_s_curve
)

sns.set_style("whitegrid")

In [ ]:
class BidirectionalMLP(nn.Module):
    def __init__(self, input_dim=2, hidden_dim=128, time_embed_dim=32, dir_embed_dim=16):
        super(BidirectionalMLP, self).__init__()

        self.time_embed = nn.Sequential(
            nn.Linear(1, time_embed_dim),
            nn.SiLU(),
            nn.Linear(time_embed_dim, time_embed_dim)
        )
        self.dir_embed = nn.Embedding(2, dir_embed_dim)

        self.net = nn.Sequential(
            nn.Linear(input_dim + time_embed_dim + dir_embed_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, input_dim)
        )

    def forward(self, x, t, s):
        if t.dim() == 1:
            t = t.unsqueeze(-1)
        if s.dim() == 2:
            s = s.squeeze(-1)

        t_emb = self.time_embed(t)
        s_emb = self.dir_embed(s.long())

        h = torch.cat([x, t_emb, s_emb], dim=1)
        return self.net(h)

class Law_class(object):
    def __init__(self, pi, device):
        self.device = device
        self.data = torch.as_tensor(pi, dtype=torch.float32).to(self.device)
        self.dim = self.data.shape[-1]

    def sample(self, size):
        indices = torch.randint(0, len(self.data), (size,))
        X = self.data[indices]
        return X.to(self.device)


In [ ]:
class Schrodinger_Bridge_Matching(object):
    def __init__(self, Law_0, Law_1, N_pretraining, N_finetuning, Bs, steps, eps,
                 pretrain_lr, finetune_lr, decay, use_ema_for_sampling=True):
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.Law_0 = Law_class(Law_0, device=self.device)
        self.Law_1 = Law_class(Law_1, device=self.device)

        self.T = 1.0
        self.N_pretraining = N_pretraining
        self.N_finetuning = N_finetuning
        self.criterion = nn.MSELoss()

        self.Bs = Bs
        self.bs = int(Bs / 2)
        self.dim = self.Law_0.dim

        self.steps = steps
        self.eps = eps
        self.pretrain_lr = pretrain_lr
        self.finetune_lr = finetune_lr
        self.decay = decay

        self.use_ema = use_ema_for_sampling

        self.delta_t = self.T / self.steps
        n_steps = int(round(self.T / self.delta_t))
        self.t_list = [torch.tensor([t], device=self.device) for t in torch.linspace(0.0, self.T, n_steps + 1)]

        self.v_theta = BidirectionalMLP(input_dim=self.dim).to(self.device)

        self.ema_params = {
            name: param.clone().detach()
            for name, param in self.v_theta.named_parameters()
        }
        for param in self.ema_params.values():
            param.requires_grad = False

        self.loss_history = {
            'pretrain_total': [],
            'pretrain_forward': [],
            'pretrain_backward': [],
            'finetune_total': [],
            'finetune_forward': [],
            'finetune_backward': []
        }

    def update_ema(self):
        with torch.no_grad():
            for name, param in self.v_theta.named_parameters():
                self.ema_params[name].mul_(self.decay).add_(param.data, alpha=1 - self.decay)

    def get_ema_model(self):
        model = copy.deepcopy(self.v_theta)
        model.eval()
        with torch.no_grad():
            for name, param in model.named_parameters():
                param.copy_(self.ema_params[name])
        return model

    def pretrain_bridge(self):
        self.v_theta.train()
        optimizer = torch.optim.Adam(self.v_theta.parameters(), lr=self.pretrain_lr)

        forward_dir = torch.ones(self.bs, dtype=torch.long, device=self.device)
        backward_dir = torch.zeros(self.Bs - self.bs, dtype=torch.long, device=self.device)

        with tqdm(total=self.N_pretraining, desc="Pretraining") as pbar:
            for n in range(self.N_pretraining):
                X0 = self.Law_0.sample(size=self.Bs)
                X1 = self.Law_1.sample(size=self.Bs)

                t = torch.rand(self.Bs, 1, device=self.device)
                Z = torch.randn(self.Bs, self.dim, device=self.device)

                Xt = (1 - t) * X0 + t * X1 + torch.sqrt(self.eps * t * (1 - t)) * Z

                optimizer.zero_grad()

                v_forward = self.v_theta(Xt[:self.bs], t[:self.bs], forward_dir)
                target_forward = (X1[:self.bs] - Xt[:self.bs]) / torch.clamp(1 - t[:self.bs], min=1e-5)
                forward_loss = self.criterion(v_forward, target_forward)

                v_backward = self.v_theta(Xt[self.bs:], 1 - t[self.bs:], backward_dir)
                target_backward = (X0[self.bs:] - Xt[self.bs:]) / torch.clamp(t[self.bs:], min=1e-5)
                backward_loss = self.criterion(v_backward, target_backward)

                loss = 0.5 * (forward_loss + backward_loss)
                loss.backward()

                grad_norm = torch.nn.utils.clip_grad_norm_(self.v_theta.parameters(), max_norm=1.0)
                optimizer.step()
                self.update_ema()

                if n % 10 == 0:
                    self.loss_history['pretrain_total'].append(loss.item())
                    self.loss_history['pretrain_forward'].append(forward_loss.item())
                    self.loss_history['pretrain_backward'].append(backward_loss.item())

                pbar.update(1)
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'fwd': f'{forward_loss.item():.4f}',
                    'bwd': f'{backward_loss.item():.4f}'
                })

    def finetune_bridge(self):
        self.v_theta.train()
        optimizer = torch.optim.Adam(self.v_theta.parameters(), lr=self.finetune_lr)

        forward_dir = torch.ones(self.bs, dtype=torch.long, device=self.device)
        backward_dir = torch.zeros(self.bs, dtype=torch.long, device=self.device)

        with tqdm(total=self.N_finetuning, desc="Finetuning") as pbar:
            for n in range(self.N_finetuning):
                X0 = self.Law_0.sample(size=self.bs)
                X1 = self.Law_1.sample(size=self.bs)

                sample_model = self.get_ema_model() if self.use_ema else self.v_theta
                sample_model.eval()

                with torch.no_grad():
                    X1_tilde = X0.clone()
                    for t in self.t_list[:-1]:
                        t_tensor = t.expand(X0.shape[0], 1)
                        drift = sample_model(X1_tilde, t_tensor, forward_dir)
                        drift = torch.clamp(drift, min=-10.0, max=10.0)
                        X1_tilde = X1_tilde + self.delta_t * drift + np.sqrt(self.eps * self.delta_t) * torch.randn_like(X1_tilde)

                    X0_tilde = X1.clone()
                    for t in self.t_list[:-1]:
                        t_tensor = t.expand(X1.shape[0], 1)
                        drift = sample_model(X0_tilde, t_tensor, backward_dir)
                        drift = torch.clamp(drift, min=-10.0, max=10.0)
                        X0_tilde = X0_tilde + self.delta_t * drift + np.sqrt(self.eps * self.delta_t) * torch.randn_like(X0_tilde)

                t_f = torch.rand(self.bs, 1, device=self.device)
                Z_f = torch.randn(self.bs, self.dim, device=self.device)
                t_b = torch.rand(self.bs, 1, device=self.device)
                Z_b = torch.randn(self.bs, self.dim, device=self.device)

                Xt_f = (1 - t_f) * X0_tilde + t_f * X1 + torch.sqrt(self.eps * t_f * (1 - t_f)) * Z_f
                Xt_b = (1 - t_b) * X0 + t_b * X1_tilde + torch.sqrt(self.eps * t_b * (1 - t_b)) * Z_b

                optimizer.zero_grad()

                v_forward = self.v_theta(Xt_f, t_f, forward_dir)
                target_forward = (X1 - Xt_f) / torch.clamp(1 - t_f, min=1e-5)
                forward_loss = self.criterion(v_forward, target_forward)

                v_backward = self.v_theta(Xt_b, 1 - t_b, backward_dir)
                target_backward = (X0 - Xt_b) / torch.clamp(t_b, min=1e-5)
                backward_loss = self.criterion(v_backward, target_backward)

                loss = 0.5 * (forward_loss + backward_loss)
                loss.backward()

                torch.nn.utils.clip_grad_norm_(self.v_theta.parameters(), max_norm=1.0)
                optimizer.step()
                self.update_ema()

                if n % 10 == 0:
                    self.loss_history['finetune_total'].append(loss.item())
                    self.loss_history['finetune_forward'].append(forward_loss.item())
                    self.loss_history['finetune_backward'].append(backward_loss.item())

                pbar.update(1)
                pbar.set_postfix({
                    'loss': f'{loss.item():.4f}',
                    'fwd': f'{forward_loss.item():.4f}',
                    'bwd': f'{backward_loss.item():.4f}'
                })

    def sample_noisy(self, x0, direction="forward", num_steps=None):
        model = self.get_ema_model()
        model.eval()

        if num_steps is None:
            num_steps = self.steps

        delta_t = self.T / num_steps
        t_list = [torch.tensor([t], device=self.device) for t in torch.linspace(0.0, self.T, num_steps + 1)]

        dir_idx = torch.ones(x0.shape[0], dtype=torch.long, device=self.device) if direction == 'forward' else torch.zeros(x0.shape[0], dtype=torch.long, device=self.device)

        x = x0.clone()
        trajectory = [x.clone()]

        with torch.no_grad():
            for i, t in enumerate(t_list[:-1]):
                t_tensor = t.expand(x.shape[0], 1)
                drift = model(x, t_tensor, dir_idx)
                x = x + delta_t * drift + np.sqrt(self.eps * delta_t) * torch.randn_like(x)
                trajectory.append(x.clone())

        return torch.stack(trajectory, dim=1)

    def sample_ode(self, x0, direction="forward", num_steps=None):
        model = self.get_ema_model()
        model.eval()

        if num_steps is None:
            num_steps = self.steps

        delta_t = self.T / num_steps
        t_list = [torch.tensor([t], device=self.device) for t in torch.linspace(0.0, self.T, num_steps + 1)]

        ones = torch.ones(x0.shape[0], dtype=torch.long, device=self.device)
        zeros = torch.zeros(x0.shape[0], dtype=torch.long, device=self.device)

        x = x0.clone()
        trajectory = [x.clone()]

        with torch.no_grad():
            for i, t in enumerate(t_list[:-1]):
                t_tensor = t.expand(x.shape[0], 1)
                drift_f = model(x, t_tensor, ones)
                drift_b = model(x, 1 - t_tensor, zeros)
                drift_ode = 0.5 * (drift_b - drift_f)

                x = x + delta_t * drift_ode
                trajectory.append(x.clone())

        return torch.stack(trajectory, dim=1)

    def sample(self, x0, method='ode', direction=None, num_steps=None):
        if method == 'ode':
            return self.sample_ode(x0, num_steps=num_steps)
        else:
            return self.sample_noisy(x0, direction=direction, num_steps=num_steps)

    def plot_losses(self, save_path='loss_curves.png'):
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        if len(self.loss_history['pretrain_total']) > 0:
            x_pre = np.arange(len(self.loss_history['pretrain_total'])) * 10
            axes[0].plot(x_pre, self.loss_history['pretrain_total'], label='Total', linewidth=2)
            axes[0].plot(x_pre, self.loss_history['pretrain_forward'], label='Forward', alpha=0.7)
            axes[0].plot(x_pre, self.loss_history['pretrain_backward'], label='Backward', alpha=0.7)
            axes[0].set_xlabel('Epoch', fontsize=12)
            axes[0].set_ylabel('Loss', fontsize=12)
            axes[0].set_title('Pretraining Loss', fontsize=14, fontweight='bold')
            axes[0].legend()
            axes[0].grid(True, alpha=0.3)
            axes[0].set_yscale('log')

        if len(self.loss_history['finetune_total']) > 0:
            x_fine = np.arange(len(self.loss_history['finetune_total'])) * 10
            axes[1].plot(x_fine, self.loss_history['finetune_total'], label='Total', linewidth=2)
            axes[1].plot(x_fine, self.loss_history['finetune_forward'], label='Forward', alpha=0.7)
            axes[1].plot(x_fine, self.loss_history['finetune_backward'], label='Backward', alpha=0.7)
            axes[1].set_xlabel('Epoch', fontsize=12)
            axes[1].set_ylabel('Loss', fontsize=12)
            axes[1].set_title('Finetuning Loss', fontsize=14, fontweight='bold')
            axes[1].legend()
            axes[1].grid(True, alpha=0.3)
            axes[1].set_yscale('log')

        plt.tight_layout()
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved loss curves to {save_path}")
        plt.close()

In [ ]:
def generate_complex_synthetic_data(experiment="blobs_to_moons", n_samples=5000, random_state=42):
    if experiment == "blobs_to_moons":
        X0, _ = make_blobs(n_samples=n_samples, centers=[[-2, -2], [2, 2]],
                           cluster_std=0.5, random_state=random_state)
        X1, _ = make_moons(n_samples=n_samples, noise=0.05, random_state=random_state)
        X1 = X1 * 2 - np.array([0.5, 0.25])

    elif experiment == "swiss_roll_to_moons":
        X0, _ = make_swiss_roll(n_samples=n_samples, noise=0.3, random_state=random_state)
        X0 = X0[:, [0, 2]] / 5
        X1, _ = make_moons(n_samples=n_samples, noise=0.05, random_state=random_state)
        X1 = X1 * 2

    elif experiment == "circles_to_blobs":
        X0, _ = make_circles(n_samples=n_samples, noise=0.05, factor=0.5, random_state=random_state)
        X0 = X0 * 3
        X1, _ = make_blobs(n_samples=n_samples, centers=[[-2, 0], [2, 0]],
                           cluster_std=0.4, random_state=random_state)

    elif experiment == "s_curve_to_spiral":
        X0, _ = make_s_curve(n_samples=n_samples, noise=0.1, random_state=random_state)
        X0 = X0[:, [0, 2]] / 2

        theta = np.sqrt(np.random.RandomState(random_state).rand(n_samples)) * 3 * np.pi
        r = theta
        X1 = np.column_stack([r * np.cos(theta), r * np.sin(theta)])
        X1 = X1 / 5

    elif experiment == "grid_to_circle":
        sqrt_n = int(np.sqrt(n_samples))
        x = np.linspace(-2, 2, sqrt_n)
        y = np.linspace(-2, 2, sqrt_n)
        xx, yy = np.meshgrid(x, y)
        X0 = np.column_stack([xx.ravel(), yy.ravel()])[:n_samples]

        angles = np.random.RandomState(random_state).uniform(0, 2*np.pi, n_samples)
        radius = 2
        X1 = np.column_stack([radius * np.cos(angles), radius * np.sin(angles)])

    elif experiment == "moons_to_circles":
        X0, _ = make_moons(n_samples=n_samples, noise=0.05, random_state=random_state)
        X0 = X0 * 2
        X1, _ = make_circles(n_samples=n_samples, noise=0.05, factor=0.5, random_state=random_state)
        X1 = X1 * 2.5

    else:
        raise ValueError(f"Unknown experiment: {experiment}")

    return X0.astype(np.float32), X1.astype(np.float32)



def plot_distributions_comparison(X0, X1, X_sde, save_path='distributions.png'):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    titles = ['Law_0 (Source)', 'Law_1 (Target)', 'Generated (SDE/Noisy)']
    data = [X0, X1, X_sde]
    colors = ['blue', 'red', 'green']

    for i in range(3):
        axes[i].scatter(data[i][:, 0], data[i][:, 1], alpha=0.5, s=20, c=colors[i])
        axes[i].set_title(titles[i], fontsize=14, fontweight='bold')
        axes[i].set_xlim(-4, 4)
        axes[i].set_ylim(-4, 4)
        axes[i].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    print(f"Saved distribution comparison to {save_path}")
    plt.close()

In [ ]:
def main():
    print("="*70)
    print("Schrödinger Bridge Training - All Experiments")
    print("="*70)

    torch.manual_seed(42)
    np.random.seed(42)

    experiments = [
        "blobs_to_moons",
        "swiss_roll_to_moons",
        "circles_to_blobs",
        "s_curve_to_spiral",
        "grid_to_circle",
        "moons_to_circles"
    ]

    for experiment in experiments:
        print(f"\n{'='*70}")
        print(f"Running experiment: {experiment}")
        print(f"{'='*70}")

        print("\n[1/6] Generating synthetic data...")
        X0, X1 = generate_complex_synthetic_data(experiment=experiment, n_samples=5000)
        print(f"   Law_0 shape: {X0.shape}")
        print(f"   Law_1 shape: {X1.shape}")

        print("\n[2/6] Initializing Schrödinger Bridge...")
        bridge = Schrodinger_Bridge_Matching(
            Law_0=X0,
            Law_1=X1,
            N_pretraining=10000,
            N_finetuning=5000,
            Bs=256,
            steps=100,
            eps=0.25,
            pretrain_lr=3e-4,
            finetune_lr=1e-5,
            decay=0.999,
            use_ema_for_sampling=True
        )

        print("\n[3/6] Pretraining the bridge...")
        bridge.pretrain_bridge()

        print("\n[4/6] Finetuning the bridge...")
        bridge.finetune_bridge()

        print("\n[5/6] Generating samples...")
        n_samples = 1000
        num_steps = 100

        X0_samples = torch.tensor(X0[:n_samples], device=bridge.device)
        trajectories_sde = bridge.sample(X0_samples, method='noisy', direction='forward', num_steps=num_steps)
        X1_generated_sde = trajectories_sde[:, -1, :].cpu().numpy()

        print("\n[6/6] Creating visualizations...")
        plot_distributions_comparison(
            X0[:n_samples],
            X1[:n_samples],
            X1_generated_sde,
            save_path=f'{experiment}_distributions_comparison.png'
        )

        bridge.plot_losses(save_path=f'{experiment}_loss_curves.png')

        print(f"\nCompleted {experiment}!")
        print(f"Generated files:")
        print(f"  - {experiment}_distributions_comparison.png")
        print(f"  - {experiment}_loss_curves.png")

        if len(bridge.loss_history['pretrain_total']) > 0:
            print(f"  - Final pretrain loss:  {bridge.loss_history['pretrain_total'][-1]:.6f}")
        if len(bridge.loss_history['finetune_total']) > 0:
            print(f"  - Final finetune loss:  {bridge.loss_history['finetune_total'][-1]:.6f}")

    print("\n" + "="*70)
    print("All Experiments Complete!")
    print("="*70)


if __name__ == "__main__":
    main()

Schrödinger Bridge Training - All Experiments

Running experiment: blobs_to_moons

[1/6] Generating synthetic data...
   Law_0 shape: (5000, 2)
   Law_1 shape: (5000, 2)

[2/6] Initializing Schrödinger Bridge...

[3/6] Pretraining the bridge...


Pretraining: 100%|██████████| 10000/10000 [01:06<00:00, 150.34it/s, loss=3.6940, fwd=2.8190, bwd=4.5689]



[4/6] Finetuning the bridge...


Finetuning: 100%|██████████| 5000/5000 [08:57<00:00,  9.31it/s, loss=1.3324, fwd=1.5034, bwd=1.1614]



[5/6] Generating samples...

[6/6] Creating visualizations...
Saved distribution comparison to blobs_to_moons_distributions_comparison.png
Saved loss curves to blobs_to_moons_loss_curves.png

Completed blobs_to_moons!
Generated files:
  - blobs_to_moons_distributions_comparison.png
  - blobs_to_moons_loss_curves.png
  - Final pretrain loss:  3.557386
  - Final finetune loss:  0.999093

Running experiment: swiss_roll_to_moons

[1/6] Generating synthetic data...
   Law_0 shape: (5000, 2)
   Law_1 shape: (5000, 2)

[2/6] Initializing Schrödinger Bridge...

[3/6] Pretraining the bridge...


...

Finetuning: 100%|██████████| 5000/5000 [09:00<00:00,  9.25it/s, loss=1.3055, fwd=0.9825, bwd=1.6285]



[5/6] Generating samples...

[6/6] Creating visualizations...
Saved distribution comparison to s_curve_to_spiral_distributions_comparison.png
Saved loss curves to s_curve_to_spiral_loss_curves.png

Completed s_curve_to_spiral!
Generated files:
  - s_curve_to_spiral_distributions_comparison.png
  - s_curve_to_spiral_loss_curves.png
  - Final pretrain loss:  3.837127
  - Final finetune loss:  0.853456

Running experiment: grid_to_circle

[1/6] Generating synthetic data...
   Law_0 shape: (4900, 2)
   Law_1 shape: (5000, 2)

[2/6] Initializing Schrödinger Bridge...

[3/6] Pretraining the bridge...


Pretraining: 100%|██████████| 10000/10000 [01:06<00:00, 150.37it/s, loss=3.4995, fwd=3.7314, bwd=3.2676]



[4/6] Finetuning the bridge...


Finetuning:  58%|█████▊    | 2902/5000 [05:19<03:35,  9.74it/s, loss=0.9151, fwd=0.5454, bwd=1.2849]